# Plaivorb: Real-time Geo-Semantic Change Detection with MariaDB

This Jupyter Notebook demonstrates the core functionalities of **Plaivorb**, a system leveraging MariaDB's advanced features (VECTOR, SPATIAL, Temporal Tables) for real-time geo-semantic change detection.

We will cover:
1.  **Setup**: Database connection and initial state.
2.  **Data Ingestion**: Simulating incoming raw sensor data.
3.  **Semantic Processing**: Transforming raw data into geo-semantic features with embeddings.
4.  **Temporal Table Demonstration**: Querying historical states of features.
5.  **MariaDB Vector Demonstration**: Showing semantic similarity and distance.
6.  **Spatial Function Demonstration**: Geospatial queries.
7.  **Change Detection**: Running the core logic and viewing results.
8.  **Visualization**: Mapping detected changes.
9.  **ColumnStore (Analytical) Query**: Example of querying historical aggregates.

In [ ]:
# 1. Initial Setup: Import necessary libraries and establish DB connection
import sys
import os
import json
import pandas as pd
import geopandas as gpd
from shapely.wkt import loads as wkt_loads
import folium
import time

# Add src directory to Python path to import modules
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '../src')))

from utils import get_db_connection
from ingest_data import ingest_raw_sensor_data
from semantic_processor import process_raw_data_into_geo_features
from change_detector import run_change_detection

print("Connecting to MariaDB...")
conn = get_db_connection()
if conn:
    print("Successfully connected to MariaDB!")
    cursor = conn.cursor(dictionary=True) # For fetching results as dictionaries
else:
    print("Failed to connect. Please check your config.ini and MariaDB server.")
    sys.exit(1)

# Function to run SQL and return as Pandas DataFrame
def run_sql_query(sql_query):
    try:
        cursor.execute(sql_query)
        columns = [i[0] for i in cursor.description]
        data = cursor.fetchall()
        return pd.DataFrame(data, columns=columns)
    except Exception as e:
        print(f"Error executing query: {e}")
        return None

# Function to run SQL and return as GeoPandas GeoDataFrame
def run_spatial_query(sql_query, geometry_column='geometry'):
    df = run_sql_query(sql_query)
    if df is not None and geometry_column in df.columns:
        # Convert WKT string to shapely geometry
        df['geometry'] = df[geometry_column].apply(wkt_loads)
        # Create a GeoDataFrame
        gdf = gpd.GeoDataFrame(df, geometry='geometry', crs="EPSG:4326")
        return gdf
    return None


## 2. Initial Data Ingestion & State Check

Let's start by simulating some initial raw sensor data ingestion. This will populate our `raw_sensor_data` table.

In [ ]:
# Clear previous detected changes for a fresh demo run
run_sql_query("DELETE FROM detected_changes;")
conn.commit()
print("Cleared detected_changes table.")

# Run initial ingestion
ingest_raw_sensor_data(conn, num_records=5)

print("\n--- Raw Sensor Data (first 5) ---")
raw_df = run_sql_query("SELECT id, timestamp, latitude, longitude, processed_status, feature_type FROM raw_sensor_data LIMIT 5;")
display(raw_df)

print("\n--- Geo Features (should be empty initially, or very few if schema setup ran) ---")
geo_df = run_sql_query("SELECT id, name, feature_type, ROW_START FROM geo_features FOR SYSTEM_TIME AS OF NOW() LIMIT 5;")
display(geo_df)

## 3. Semantic Processing

Now, let's process the raw data. This step generates synthetic semantic embeddings and creates/updates records in our `geo_features` table. Each feature will now have a `VECTOR` embedding and `GEOMETRY`.

In [ ]:
process_raw_data_into_geo_features(conn)

print("\n--- Processed Geo Features (first 5) ---")
geo_df_processed = run_sql_query("SELECT id, name, feature_type, semantic_embedding, ST_AsText(geometry) as geometry_wkt, ROW_START FROM geo_features FOR SYSTEM_TIME AS OF NOW() LIMIT 5;")
display(geo_df_processed)

print("\n--- Raw Sensor Data (status should be PROCESSED) ---")
raw_df_processed = run_sql_query("SELECT id, processed_status FROM raw_sensor_data WHERE processed_status = 'PROCESSED' LIMIT 5;")
display(raw_df_processed)

## 4. Temporal Table Demonstration

MariaDB's Temporal Tables allow us to query the state of our `geo_features` table at any point in time. Let's see how `ROW_START` changes when a feature is updated.

First, we'll introduce some new raw data, process it, and then query the historical states.

In [ ]:
print("\n--- Ingesting MORE raw data to trigger updates/new features ---")
ingest_raw_sensor_data(conn, num_records=3)
process_raw_data_into_geo_features(conn)

time.sleep(1) # Ensure enough time passes for a new ROW_START timestamp if an update occurs

print("\n--- All versions of a specific geo_feature (showing temporal history) ---")
try:
    # Get an ID of an existing feature that might have been updated
    first_feature_id = run_sql_query("SELECT id FROM geo_features LIMIT 1;").iloc[0]['id']
    print(f"Fetching temporal history for feature ID: {first_feature_id}")
    temporal_df = run_sql_query(f"SELECT id, name, feature_type, semantic_embedding, ROW_START, ROW_END FROM geo_features FOR SYSTEM_TIME ALL WHERE id = {first_feature_id} ORDER BY ROW_START;")
    display(temporal_df)

    print("\n--- Querying geo_features 'AS OF' a past timestamp ---")
    # Query a version from the past (e.g., 2 minutes ago)
    past_time = datetime.now() - timedelta(minutes=2)
    past_time_str = past_time.strftime("%Y-%m-%d %H:%M:%S")
    as_of_df = run_sql_query(f"SELECT id, name, feature_type, ROW_START FROM geo_features FOR SYSTEM_TIME AS OF '{past_time_str}' WHERE id = {first_feature_id};")
    display(as_of_df)
except IndexError:
    print("Not enough features to demonstrate temporal queries effectively.")

## 5. MariaDB Vector Demonstration

The `VECTOR` data type is central to Plaivorb's semantic change detection. We'll demonstrate `VECTOR_DISTANCE` to compare embeddings.

In [ ]:
print("\n--- Comparing Semantic Embeddings using VECTOR_DISTANCE ---")

# Fetch two feature embeddings to compare
embeddings_df = run_sql_query("SELECT id, name, semantic_embedding FROM geo_features FOR SYSTEM_TIME AS OF NOW() LIMIT 2;")

if len(embeddings_df) == 2:
    id1, name1, emb1_str = embeddings_df.iloc[0].values
    id2, name2, emb2_str = embeddings_df.iloc[1].values

    # Note: MariaDB returns VECTOR as a JSON string, so we need to pass it back as such
    distance_query = f"SELECT VECTOR_DISTANCE('{emb1_str}', '{emb2_str}') AS semantic_distance;"
    distance_df = run_sql_query(distance_query)

    print(f"Semantic Distance between '{name1}' (ID: {id1}) and '{name2}' (ID: {id2}):")
    display(distance_df)

    # Now, let's artificially change an embedding and re-compare to show increased distance
    print("\n--- Artificially changing an embedding to show greater distance ---")
    original_embedding_list = json.loads(emb1_str)
    # Create a significantly different embedding
    changed_embedding_list = [(x + 0.5) % 1.0 for x in original_embedding_list] # Shift values
    changed_embedding_str = json.dumps(changed_embedding_list)

    distance_changed_query = f"SELECT VECTOR_DISTANCE('{emb1_str}', '{changed_embedding_str}') AS semantic_distance_changed;"
    distance_changed_df = run_sql_query(distance_changed_query)
    print(f"Semantic Distance between original '{name1}' and a significantly changed version:")
    display(distance_changed_df)
else:
    print("Not enough features to demonstrate VECTOR_DISTANCE effectively. Please run ingestion/processing again.")

## 6. Spatial Function Demonstration

MariaDB's spatial functions (`ST_Contains`, `ST_Intersects`, `ST_Area`, etc.) are essential for Plaivorb. Let's run some basic spatial queries.

In [ ]:
print("\n--- Demonstrating Spatial Queries ---")

# Find features within a bounding box (example: a small area)
bbox_query = "SELECT id, name, feature_type, ST_AsText(geometry) AS geometry_wkt FROM geo_features WHERE ST_Within(geometry, ST_GeomFromText('POLYGON((-118.25 34.0, -118.25 34.05, -118.2 34.05, -118.2 34.0, -118.25 34.0))', 4326));"
bbox_df = run_spatial_query(bbox_query, geometry_column='geometry_wkt')
print("Features within a specific bounding box:")
display(bbox_df)

# Calculate area of features
area_query = "SELECT id, name, feature_type, ST_Area(geometry) AS area_sq_degrees FROM geo_features LIMIT 5;"
area_df = run_sql_query(area_query)
print("\nArea of features:")
display(area_df)


## 7. Change Detection Execution

Now, let's run the `DetectFeatureChange` stored procedure (via `ProcessAllFeaturesForChangeDetection`) to identify actual semantic and geometric changes based on our configured thresholds.

We'll first ingest and process new data to ensure there's a chance for changes to be detected.

In [ ]:
print("\n--- Ingesting more data and processing to create more potential changes ---")
ingest_raw_sensor_data(conn, num_records=5) # More data, more chances for updates/new features
process_raw_data_into_geo_features(conn)

time.sleep(2) # Give a moment for timestamps to differ for temporal queries

print("\n--- Running the Plaivorb Change Detection --- ")
# Adjust semantic_threshold as needed based on your synthetic data variations
run_change_detection(conn, semantic_threshold=0.08, geometry_tolerance=0.001)

print("\n--- All Detected Changes in `detected_changes` table ---")
detected_changes_df = run_spatial_query("SELECT id, change_timestamp, feature_id, feature_name, change_type, severity, details, ST_AsText(location) as geometry_wkt FROM detected_changes ORDER BY change_timestamp DESC;")
display(detected_changes_df)

## 8. Visualization of Detected Changes

Let's visualize the detected changes on an interactive map using `folium`.

In [ ]:
if detected_changes_df is not None and not detected_changes_df.empty:
    # Create a base map centered around the data
    center_lat = detected_changes_df.geometry.y.mean()
    center_lon = detected_changes_df.geometry.x.mean()
    m = folium.Map(location=[center_lat, center_lon], zoom_start=14)

    for idx, row in detected_changes_df.iterrows():
        popup_html = f"<b>Change Type:</b> {row['change_type']}<br>"
        popup_html += f"<b>Feature:</b> {row['feature_name']} (ID: {row['feature_id']})<br>"
        popup_html += f"<b>Severity:</b> {row['severity']}<br>"
        popup_html += f"<b>Timestamp:</b> {row['change_timestamp']}<br>"
        if row['details']:
             popup_html += f"<b>Details:</b> {json.loads(row['details'])}"
        
        # For polygons, use add_child(folium.GeoJson) for proper rendering
        if row.geometry.geom_type == 'Point':
            folium.Marker(
                location=[row.geometry.y, row.geometry.x],
                popup=folium.Popup(popup_html, max_width=300),
                icon=folium.Icon(color='red', icon='info-sign')
            ).add_to(m)
        elif row.geometry.geom_type == 'Polygon':
             folium.GeoJson(
                data=json.loads(gpd.GeoSeries([row.geometry]).to_json())['features'][0]['geometry'],
                style_function=lambda x: {'fillColor': '#FF0000', 'color': '#FF0000', 'weight': 2, 'fillOpacity': 0.4},
                tooltip=folium.Tooltip(popup_html)
            ).add_to(m)

    display(m)
else:
    print("No changes to visualize or DataFrame is empty.")

## 9. ColumnStore (Analytical) Query Example

Demonstrates how `ColumnStore` can be used for high-performance analytical queries on historical or aggregated geo-semantic data. We'll populate some synthetic data and then query it.

In [ ]:
print("\n--- Populating Synthetic ColumnStore Data ---")
for i in range(10):
    snapshot_date = (datetime.now() - timedelta(days=i*30)).strftime('%Y-%m-%d')
    feature_type = random.choice(['building', 'road', 'forest', 'water'])
    avg_semantic_value = random.uniform(0.1, 0.9)
    count_features = random.randint(100, 10000)
    total_area = random.uniform(1000, 50000)
    try:
        cursor.execute(
            "INSERT INTO historical_geo_summary (snapshot_date, feature_type, avg_semantic_value, count_features, total_area) VALUES (?, ?, ?, ?, ?)",
            (snapshot_date, feature_type, avg_semantic_value, count_features, total_area)
        )
    except mariadb.Error as e:
        if "Duplicate entry" in str(e): # Handle potential duplicate primary key if running multiple times
            print(f"  Skipping duplicate entry for {snapshot_date}, {feature_type}")
        else:
            print(f"  Error inserting ColumnStore data: {e}")

conn.commit()
print("Synthetic ColumnStore data populated.")

print("\n--- Analytical Query on ColumnStore: Total area per feature type over last 6 months ---")
columnstore_query = """
SELECT
    feature_type,
    SUM(total_area) AS total_area_sq_degrees,
    AVG(avg_semantic_value) AS avg_semantic_val
FROM historical_geo_summary
WHERE snapshot_date >= (CURDATE() - INTERVAL 6 MONTH)
GROUP BY feature_type
ORDER BY total_area_sq_degrees DESC;
"""
columnstore_df = run_sql_query(columnstore_query)
display(columnstore_df)


In [ ]:
# Clean up database connection
if conn:
    conn.close()
    print("\nMariaDB connection closed.")